Loading and imports

In [1]:
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
sys.path.insert(0, str(PROJECT_ROOT))

from src.features.temporal import build_temporal_features
from src.features.order_features import build_order_features
from src.data.build_dataset import build_targets

DATA_PATH = PROJECT_ROOT / "data" / "raw"

orders = pd.read_csv(
    DATA_PATH / "olist_orders_dataset.csv",
    parse_dates=[
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ],
)

order_items = pd.read_csv(
    DATA_PATH / "olist_order_items_dataset.csv"
)

Building the feature tables

In [2]:
temporal_features = build_temporal_features(orders)

order_features = build_order_features(order_items)

targets = build_targets(orders)

print("Temporal:", temporal_features.shape)
print("Order features:", order_features.shape)
print("Targets:", targets.shape)

Temporal: (99441, 6)
Order features: (98666, 6)
Targets: (96470, 3)


Integrate the data

In [3]:
dataset = (
    targets
    .merge(
        temporal_features,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
    .merge(
        order_features,
        on="order_id",
        how="left",
        validate="one_to_one",
    )
)

print("Dataset:", dataset.shape)

display(dataset.head())

Dataset: (96470, 13)


,order_id,delivery_time_days,is_delayed,purchase_month,purchase_weekday,purchase_hour,purchase_year,purchase_is_weekend,items_quantity,unique_products,sellers_quantity,total_price,total_freight
0,e481f51cbdc54678b7cc49136f2d6af7,8.436574,0,10,0,10,2017,0,1,1,1,29.99,8.72
1,53cdb2fc8bc7dce0b6741e2150273451,13.782037,0,7,1,20,2018,0,1,1,1,118.70,22.76
2,47770eb9100c2d0c44946d9cf07ec65d,9.394213,0,8,2,8,2018,0,1,1,1,159.90,19.22
3,949d5b44dbf5de918fe9c16f97b45f8a,13.208750,0,11,5,19,2017,1,1,1,1,45.00,27.20
4,ad21c59c0840e6cb83a9ceb5573f8159,2.873877,0,2,1,21,2018,0,1,1,1,19.90,8.72


Integrity tests

In [4]:
assert dataset["order_id"].is_unique

assert len(dataset) == len(targets)

assert dataset["delivery_time_days"].notna().all()

assert (dataset["delivery_time_days"] >= 0).all()

assert dataset["is_delayed"].isin([0, 1]).all()

assert dataset["purchase_month"].between(1, 12).all()

assert dataset["purchase_weekday"].between(0, 6).all()

assert dataset["purchase_hour"].between(0, 23).all()

assert (dataset["items_quantity"] >= 1).all()

assert (dataset["sellers_quantity"] >= 1).all()

print("All validations passed.")

All validations passed.


Audit of missing values

In [5]:
missing_report = (
    dataset.isna()
    .sum()
    .to_frame("missing_count")
)

missing_report["missing_percentage"] = (
    missing_report["missing_count"]
    / len(dataset)
    * 100
)

display(
    missing_report.sort_values(
        "missing_count",
        ascending=False,
    )
)

,missing_count,missing_percentage
order_id,0,0.0
delivery_time_days,0,0.0
is_delayed,0,0.0
purchase_month,0,0.0
purchase_weekday,0,0.0
purchase_hour,0,0.0
purchase_year,0,0.0
purchase_is_weekend,0,0.0
items_quantity,0,0.0
unique_products,0,0.0


Load the geographic dataset

In [6]:
from src.features.geographic import build_geographic_features

customers = pd.read_csv(
    DATA_PATH / "olist_customers_dataset.csv"
)

sellers = pd.read_csv(
    DATA_PATH / "olist_sellers_dataset.csv"
)

geolocation = pd.read_csv(
    DATA_PATH / "olist_geolocation_dataset.csv"
)

Build the features

In [7]:
geographic_features = build_geographic_features(
    orders=orders,
    order_items=order_items,
    customers=customers,
    sellers=sellers,
    geolocation=geolocation,
)

print(geographic_features.shape)

display(geographic_features.head())

(98666, 6)


,order_id,min_distance_km,avg_distance_km,max_distance_km,all_sellers_same_state,has_complete_coordinates
0,00010242fe8c5a6d1ba2dd792cb16214,301.229205,301.229205,301.229205,False,True
1,00018f77f2f0320c557190d7a144bdd3,588.404069,588.404069,588.404069,True,True
2,000229ec398224ef6ca0657da4fc703e,312.835023,312.835023,312.835023,True,True
3,00024acbcdf0a6daa1e931b038114c75,295.032480,295.032480,295.032480,True,True
4,00042b26cf59d7ce69dfabb4e55b4fd9,646.468544,646.468544,646.468544,False,True


Integrate into existing dataset

In [8]:
dataset = dataset.merge(
    geographic_features,
    on="order_id",
    how="left",
    validate="one_to_one",
)

assert len(dataset) == len(targets)
assert dataset["order_id"].is_unique

print(dataset.shape)

(96470, 18)


Audit geographic features

In [9]:
geo_columns = [
    "min_distance_km",
    "avg_distance_km",
    "max_distance_km",
    "has_complete_coordinates",
    "all_sellers_same_state",
]

display(dataset[geo_columns].describe(include="all"))

print("\nMissing Values:")
print(dataset[geo_columns].isna().sum())

print("\nCoverage of Coordinates:")
print(
    dataset["has_complete_coordinates"]
    .value_counts(dropna=False)
)

,min_distance_km,avg_distance_km,max_distance_km,has_complete_coordinates,all_sellers_same_state
count,95989.000000,95989.000000,95989.000000,96470,96470
unique,NaN,NaN,NaN,2,2
top,NaN,NaN,NaN,True,False
freq,NaN,NaN,NaN,95989,61895
mean,599.132014,600.460753,601.795244,NaN,NaN
std,592.254868,592.153529,592.651265,NaN,NaN
min,0.000000,0.000000,0.000000,NaN,NaN
25%,182.271223,186.872620,188.637540,NaN,NaN
50%,431.961097,433.897007,435.543755,NaN,NaN
75%,796.570074,797.337474,800.188904,NaN,NaN



Missing Values:
min_distance_km             481
avg_distance_km             481
max_distance_km             481
has_complete_coordinates      0
all_sellers_same_state        0
dtype: int64

Coverage of Coordinates:
has_complete_coordinates
True     95989
False      481
Name: count, dtype: int64


Adding the original timestamp as metadata

In [10]:
purchase_metadata = orders[
    ["order_id", "order_purchase_timestamp"]
].copy()

dataset = dataset.merge(
    purchase_metadata,
    on="order_id",
    how="left",
    validate="one_to_one",
)

assert dataset["order_purchase_timestamp"].notna().all()

Some additional tests

In [21]:
assert dataset["order_id"].is_unique
assert len(dataset) == 96470

distance_columns = [
    "min_distance_km",
    "avg_distance_km",
    "max_distance_km",
]

missing = dataset[distance_columns].isna()

assert not (
    missing.any(axis=1)
    & ~missing.all(axis=1)
).any(), "there are rows with distances partially missing"

valid_distances = dataset.dropna(
    subset=distance_columns
)

assert (
    valid_distances[distance_columns] >= 0
).all().all(), "there are negative distances"

tolerance = 1e-9

assert (
    valid_distances["min_distance_km"]
    <= valid_distances["avg_distance_km"] + tolerance
).all(), "min_distance_km is greater than avg_distance_km"

assert (
    valid_distances["avg_distance_km"]
    <= valid_distances["max_distance_km"] + tolerance
).all(), "avg_distance_km is greater than max_distance_km"


print("Geographic features validation passed.")

Geographic features validation passed.


### Data leakage audit

Prediction point: order placement (order_purchase_timestamp). 

Only information known at that point should enter the predictive model. The target variables are derived from delivery outcomes and must never be used as model inputs. order_id is an identifier and the original purchase timestamp is retained as metadata for chronological splitting, not as a direct predictor.

This is a schema and availability-contract audit, not a proof that upstream feature-building functions are leakage-free. In particular, order-item and seller information must be available at the prediction point, and geographic lookup tables must not incorporate future outcomes.

In [22]:
TARGET_COLUMNS = ["delivery_time_days", "is_delayed"]
METADATA_COLUMNS = ["order_id", "order_purchase_timestamp"]

FEATURE_COLUMNS = [
    "purchase_month", "purchase_weekday", "purchase_hour", "purchase_year",
    "purchase_is_weekend", "items_quantity", "unique_products", "sellers_quantity",
    "total_price", "total_freight",
    "min_distance_km", "avg_distance_km", "max_distance_km",
    "has_complete_coordinates", "all_sellers_same_state",
]

assert len(set(FEATURE_COLUMNS + TARGET_COLUMNS + METADATA_COLUMNS)) == (
    len(FEATURE_COLUMNS) + len(TARGET_COLUMNS) + len(METADATA_COLUMNS)
), "Feature, target and metadata sets overlap"

unexpected_columns = sorted(set(dataset.columns) - set(FEATURE_COLUMNS + TARGET_COLUMNS + METADATA_COLUMNS))
missing_columns = sorted(set(FEATURE_COLUMNS + TARGET_COLUMNS + METADATA_COLUMNS) - set(dataset.columns))
assert not unexpected_columns, f"Unreviewed columns: {unexpected_columns}"
assert not missing_columns, f"Missing expected columns: {missing_columns}"

assert dataset["order_purchase_timestamp"].notna().all()
assert dataset["order_id"].is_unique

X = dataset[FEATURE_COLUMNS].copy()
y = dataset[TARGET_COLUMNS].copy()
metadata = dataset[METADATA_COLUMNS].copy()

assert not set(TARGET_COLUMNS + METADATA_COLUMNS).intersection(X.columns)
assert X.index.equals(y.index) and X.index.equals(metadata.index)

print(f"Predictors: {len(FEATURE_COLUMNS)} | Targets: {len(TARGET_COLUMNS)} | Metadata: {len(METADATA_COLUMNS)}")
print("Schema-based leakage checks passed.")
print("Availability assumption: order items, prices and sellers are known at purchase time.")

Predictors: 15 | Targets: 2 | Metadata: 2
Schema-based leakage checks passed.
Availability assumption: order items, prices and sellers are known at purchase time.


### Persist the feature-engineered dataset

Persist the complete analytical dataset (predictors, targets, and metadata) to a single Parquet file. The next notebook must use FEATURE_COLUMNS or an equivalent reviewed allowlist when constructing X; it must not train on all columns automatically. Parquet requires pyarrow or fastparquet (install pyarrow if necessary).

In [24]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "03_feature_engineering.parquet"
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

dataset.to_parquet(OUTPUT_PATH, index=False, engine="pyarrow")

reloaded = pd.read_parquet(OUTPUT_PATH, engine="pyarrow")
pd.testing.assert_frame_equal(
    dataset.reset_index(drop=True),
    reloaded.reset_index(drop=True),
    check_dtype=True,
)
assert reloaded["order_id"].is_unique
assert reloaded["order_purchase_timestamp"].notna().all()

print(f"Saved: {OUTPUT_PATH}")
print(f"Rows: {len(reloaded):,} | Columns: {len(reloaded.columns)}")

Saved: /home/arthur/Desktop/delivery-intelligence/data/processed/03_feature_engineering.parquet
Rows: 96,470 | Columns: 19


### Findings and conclusions

- Dataset integrity: the notebook constructs a delivered-order dataset with 96,470 unique orders and 19 columns (15 predictors, 2 targets, 2 metadata fields). Existing assertions validate target values, time ranges, item counts, and row uniqueness.
- Geographic coverage: distance features may be missing when coordinates are incomplete; the notebook retains these orders rather than discarding them. Geographic distance ordering is checked using a 1e-9 tolerance to account for floating-point rounding.
- Leakage controls: delivery-derived targets (delivery_time_days, is_delayed) and identifiers/timestamp metadata are excluded from X using an explicit feature allowlist. This prevents accidental inclusion during training, but does not independently validate the upstream feature-generating implementations.
- Prediction-time assumptions: prices, items, seller assignments, and geographic information must be available when the order is placed. If the intended prediction time differs, revisit the feature allowlist.
- Reproducibility: the processed dataset is written to data/processed/03_feature_engineering.parquet and checked via a round-trip equality assertion, including column types.
